<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/mineral_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# mineral_mapping.py
import numpy as np
import pandas as pd
from PIL import Image
from skimage.color import rgb2lab
from sklearn.cluster import KMeans
from matplotlib.image import imsave

# === 1) Load the Perseverance image ===
# Adjust this path if needed to point at your saved Mastcam-Z PNG
src_path = "Mars_Perseverance_SIF_1476_0797970484_351EBY_N0720000SRLC00636_0000LMJ.png"
img = Image.open(src_path).convert("RGB")
arr = np.array(img)

# === 2) Convert to LAB and cluster ===
lab = rgb2lab(arr)
h, w, _ = lab.shape
lab_flat = lab.reshape((-1, 3))

km = KMeans(n_clusters=4, random_state=42).fit(lab_flat)
labels = km.labels_.reshape(h, w)

# === 3) Define mineral labels & colors ===
label_names = {
    0: "Olivine/Basaltic Sand",
    1: "Dust-Covered Regolith",
    2: "Sulfate-Rich Zone",
    3: "Clay-Bearing Outcrop"
}
colors = {
    0: (139, 69, 19),
    1: (210, 180, 140),
    2: (100, 149, 237),
    3: (178, 34, 34)
}

# === 4) Build annotated image ===
anno = np.zeros((h, w, 3), dtype=np.uint8)
for k, col in colors.items():
    anno[labels==k] = col

# Save the PNG
out_png = "Mineral_Map_Phase2_Annotated.png"
imsave(out_png, anno)
print(f"Annotated map saved to {out_png}")

# === 5) Build & save CSV ===
coords = [(y, x) for y in range(h) for x in range(w)]
classes = [label_names[labels[y, x]] for y, x in coords]
df = pd.DataFrame(coords, columns=["y","x"])
df["mineral_class"] = classes

out_csv = "Mars_Mineral_Classification_Phase2.csv"
df.to_csv(out_csv, index=False)
print(f"Pixel classification CSV saved to {out_csv}")
